# Your first fit

Fit a single-plane strong lens end to end: an elliptical power-law (EPL) deflector
with external shear and Sérsic lens light, plus a Sérsic source behind it. You will
build the model, attach one image, and run the full **MAP → SVI → HMC** inference
chain.

:::{admonition} Running this yourself
:class: note
gigalens needs a GPU and `JAX_ENABLE_X64=1` for the canonical float64 pipeline.
This tutorial is rendered from `demos/simple_demo.ipynb` — run that notebook to
reproduce the figures.
:::

In [ ]:
%matplotlib inline
import numpy as np
import jax
from jax import numpy as jnp
import optax
import tensorflow_probability.substrates.jax as tfp
import matplotlib as mpl
from matplotlib import pyplot as plt
from corner import corner
tfd = tfp.distributions

import gigalens
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.mass.shear import Shear
from gigalens.jax.profiles.light.sersic import SersicEllipse
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.scene_simulator import SceneSimulator
from gigalens.jax.scene_prob_model import ImageData, ProbModel
from gigalens.jax.inference import ModellingSequence
print('jax', jax.__version__, '| devices', jax.devices())

## Build the model

A `Component` bundles a **profile** with a dict of **priors**, one entry per free
parameter of that profile. Components are grouped into `Plane`s, and the planes make
a `LensModel`. Here the first plane carries the deflectors (EPL + external shear) and
the lens light; the second plane, placed by `deflection_ratio=1.0`, carries the source.

:::{admonition} Sampled vs solved light amplitudes
:class: tip
A light profile's amplitude is either **sampled** (`use_lstsq=False`, so the amplitude
name — here `Ie` — becomes a free parameter you give a prior) or **solved by least
squares** at render time (`use_lstsq=True`). This fit samples them; the shapelets and
multi-band tutorials use the least-squares route.
:::

In [ ]:
epl = Component(EPL(50), dict(
    theta_E=tfd.LogNormal(jnp.log(1.25), 0.25), gamma=tfd.TruncatedNormal(2, 0.25, 1, 3),
    e1=tfd.Normal(0, 0.1), e2=tfd.Normal(0, 0.1),
    center_x=tfd.Normal(0, 0.05), center_y=tfd.Normal(0, 0.05)))
shear = Component(Shear(), dict(gamma1=tfd.Normal(0, 0.05), gamma2=tfd.Normal(0, 0.05)))
lens_light = Component(SersicEllipse(use_lstsq=False), dict(
    R_sersic=tfd.LogNormal(jnp.log(1.0), 0.15), n_sersic=tfd.Uniform(2, 6),
    e1=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3), e2=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
    center_x=tfd.Normal(0, 0.05), center_y=tfd.Normal(0, 0.05), Ie=tfd.LogNormal(jnp.log(500.0), 0.3)))
source_light = Component(SersicEllipse(use_lstsq=False), dict(
    R_sersic=tfd.LogNormal(jnp.log(0.25), 0.15), n_sersic=tfd.Uniform(0.5, 4),
    e1=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5), e2=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5),
    center_x=tfd.Normal(0, 0.25), center_y=tfd.Normal(0, 0.25), Ie=tfd.LogNormal(jnp.log(150.0), 0.5)))

model = LensModel([
    Plane(mass=[epl, shear], light=[lens_light]),
    Plane(deflection_ratio=1.0, light=[source_light]),
])
print('free parameters:', model.num_free_params)

## Attach data

Load the demo image and PSF, wrap them in a `SimulatorConfig`, and build an
`ImageData`. `sees="all"` means this observation renders every light `Component` in
the model. The `ProbModel` ties the model to the data, and `ModellingSequence` is the
inference driver.

In [ ]:
root = gigalens.__path__[0]
kernel = np.load(f'{root}/assets/psf.npy').astype(np.float32)
observed_img = np.load(f'{root}/assets/demo.npy')
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=1, kernel=kernel)

background_rms, exp_time = 0.2, 100
ds = ImageData(observed_img, sim_config, background_rms=background_rms, exp_time=exp_time, sees='all')
prob = ProbModel(model, ds, mode='forward')
seq = ModellingSequence(prob)

plt.imshow(observed_img, vmin=0, vmax=10); plt.colorbar(); plt.title('Observed'); plt.show()

## Sanity check: residuals at ground truth

Before fitting, render the model at the known truth and look at the **normalized
residuals** and reduced χ². Plots before metrics: a residual image that isn't
structureless tells you the forward model or noise is wrong *before* you trust any
posterior.

In [ ]:
truth_by_path = {
    'planes/0/mass/0/theta_E': 1.1, 'planes/0/mass/0/gamma': 2.0,
    'planes/0/mass/0/e1': 0.1, 'planes/0/mass/0/e2': 0.1,
    'planes/0/mass/0/center_x': 0.1, 'planes/0/mass/0/center_y': 0.0,
    'planes/0/mass/1/gamma1': -0.01, 'planes/0/mass/1/gamma2': 0.03,
    'planes/0/light/0/R_sersic': 0.8, 'planes/0/light/0/n_sersic': 2.5,
    'planes/0/light/0/e1': 0.09534746574143645, 'planes/0/light/0/e2': 0.14849487967198177,
    'planes/0/light/0/center_x': 0.1, 'planes/0/light/0/center_y': 0.0, 'planes/0/light/0/Ie': 499.3695906504067,
    'planes/1/light/0/R_sersic': 0.25, 'planes/1/light/0/n_sersic': 1.5,
    'planes/1/light/0/e1': 0.0, 'planes/1/light/0/e2': 0.0,
    'planes/1/light/0/center_x': 0.09566681002252231, 'planes/1/light/0/center_y': -0.0639623054267272,
    'planes/1/light/0/Ie': 149.58828877085668}
names = list(model.z_param_names)
truth = np.array([truth_by_path[n] for n in names])

sim = SceneSimulator(model, sim_config)
truth_img = np.asarray(sim.simulate(model.to_params({n: jnp.asarray(truth_by_path[n]) for n in names})))
err_map = np.sqrt(background_rms**2 + np.clip(truth_img, 0, np.inf) / exp_time)
resid = (truth_img - observed_img) / err_map
print('reduced chi^2 at truth:', float(np.mean(resid**2)))

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].imshow(truth_img, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=20)); ax[0].axis('off'); ax[0].set_title('Truth')
im = ax[1].imshow(resid, cmap='coolwarm', vmin=-5, vmax=5); ax[1].axis('off'); ax[1].set_title('Normalized residual')
plt.colorbar(im, ax=ax[1]); plt.show()

## Inference: MAP → SVI → HMC

All three stages run through the same differentiable `prob.log_prob(z)`, where `z` is
the model's flat, unconstrained parameter vector: gradient descent to the MAP, a
variational fit (SVI) for a surrogate, then HMC sampling from it.

In [ ]:
def to_constrained(z_rows):
    xb = model.bijector.forward(jnp.asarray(z_rows).reshape(-1, len(names)))
    return np.stack([np.asarray(xb[n]).reshape(-1) for n in names], axis=1)

opt = optax.adabelief(1e-2, b1=0.95, b2=0.99)
best_z, best_lp, _ = seq.MAP(opt, seed=0, output_type='best')
best_z = np.asarray(jax.device_get(best_z))
print('MAP log-post: %.4g' % float(best_lp))

In [ ]:
opt = optax.adabelief(1e-4, b1=0.95, b2=0.99)
qz, loss_hist = seq.SVI(best_z, opt, n_vi=1000, num_steps=1500)
plt.plot(np.asarray(loss_hist).reshape(-1)); plt.xlabel('step'); plt.ylabel('-ELBO'); plt.title('SVI loss'); plt.show()

In [ ]:
# Rebuild the SVI surrogate from host arrays: under JAX 0.10 the sharded qz trips a
# mesh (Manual vs Explicit) clash inside HMC's pmapped sampler. device_get strips the
# sharding tag while keeping the SVI-learned mean + covariance.
qz = tfd.MultivariateNormalFullCovariance(
    loc=np.asarray(jax.device_get(qz.mean())),
    covariance_matrix=np.asarray(jax.device_get(qz.covariance())))
samples = seq.HMC(qz, num_burnin_steps=250, num_results=750, pbar_interval=25)
rhat = np.asarray(tfp.mcmc.potential_scale_reduction(samples, independent_chain_ndims=2))
ess = np.asarray(tfp.mcmc.effective_sample_size(samples, cross_chain_dims=[1, 2]))
print('max R-hat: %.3f | min ESS: %.0f' % (np.nanmax(rhat), np.nanmin(ess)))

## From `z` to physical parameters

The sampler works in the flat `z` space. Map samples back to constrained physical
values with `model.bijector.forward(...)`; column order follows `model.z_param_names`.

:::{admonition} Flat-z convention
:class: important
`model.bijector` maps a flat array of shape `(..., num_free_params)` to the constrained
parameter dict. The older list-of-columns form (`bijector.forward(list(z.T))`) is
retired — it warns and will become an error — so always pass a `(..., num_free_params)`
array.
:::

In [ ]:
n_params = samples.shape[-1]
post = to_constrained(np.asarray(samples).reshape(-1, n_params))
mass_paths = ['planes/0/mass/0/theta_E', 'planes/0/mass/0/gamma', 'planes/0/mass/0/e1',
              'planes/0/mass/0/e2', 'planes/0/mass/0/center_x', 'planes/0/mass/0/center_y',
              'planes/0/mass/1/gamma1', 'planes/0/mass/1/gamma2']
mass_labels = [r'$\theta_E$', r'$\gamma$', r'$e_1$', r'$e_2$', r'$x_{lens}$', r'$y_{lens}$',
               r'$\gamma_{1,\rm ext}$', r'$\gamma_{2,\rm ext}$']
idx = [names.index(p) for p in mass_paths]
fig = corner(post[:, idx], labels=mass_labels, truths=truth[idx],
             show_titles=True, title_fmt='.3f')
fig.suptitle('Lens mass parameters'); plt.show()